# Module 3 - Class 6 LAB: Leakage Detection and Pipelines

**Objective:** Understand data leakage and learn to prevent it using sklearn Pipelines.

### What you will learn
- Why scaling before splitting causes data leakage
- How to build proper sklearn Pipelines
- Cross-validation with pipelines

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")

## 1. Load and Prepare Data

In [ ]:
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)

# Fix TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
# Note: we intentionally leave NaN here so the pipeline can handle imputation

# Standardize service columns
replace_cols = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in replace_cols:
    df[col] = df[col].replace('No internet service', 'No')
df['MultipleLines'] = df['MultipleLines'].replace('No phone service', 'No')

# Target
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Features and target
X = df.drop(['customerID', 'Churn'], axis=1)
y = df['Churn']

numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

print(f"Features: {X.shape[1]} ({len(numeric_cols)} numeric, {len(cat_cols)} categorical)")
print(f"Target distribution:\n{y.value_counts()}")

## 2. The WRONG Way: Scale Before Split (Data Leakage)

This is a common mistake. We scale the entire dataset, then split. The scaler learns statistics (mean, std) from the test set too -- information the model should not have.

In [ ]:
# WRONG: fit scaler on ALL data (including future test data)
X_numeric_all = X[numeric_cols].copy()
X_numeric_all = X_numeric_all.fillna(X_numeric_all.median())

scaler_wrong = StandardScaler()
X_numeric_scaled_wrong = scaler_wrong.fit_transform(X_numeric_all)  # Leakage!
X_numeric_scaled_wrong = pd.DataFrame(X_numeric_scaled_wrong, columns=numeric_cols)

# Now split
X_train_wrong, X_test_wrong, y_train_w, y_test_w = train_test_split(
    X_numeric_scaled_wrong, y, test_size=0.2, random_state=42, stratify=y
)

# Train and evaluate
model_wrong = LogisticRegression(max_iter=1000, random_state=42)
model_wrong.fit(X_train_wrong, y_train_w)
y_pred_wrong = model_wrong.predict(X_test_wrong)

acc_wrong = accuracy_score(y_test_w, y_pred_wrong)
f1_wrong = f1_score(y_test_w, y_pred_wrong)
print(f"WRONG way (leakage) - Accuracy: {acc_wrong:.4f}, F1: {f1_wrong:.4f}")

## 3. The RIGHT Way: Split First, Then Scale

We split first, then fit the scaler only on training data.

In [ ]:
# RIGHT: split first
X_numeric_right = X[numeric_cols].copy()
X_numeric_right = X_numeric_right.fillna(X_numeric_right.median())

X_train_right, X_test_right, y_train_r, y_test_r = train_test_split(
    X_numeric_right, y, test_size=0.2, random_state=42, stratify=y
)

# Fit scaler ONLY on training data
scaler_right = StandardScaler()
X_train_scaled = scaler_right.fit_transform(X_train_right)
X_test_scaled = scaler_right.transform(X_test_right)  # Only transform!

# Train and evaluate
model_right = LogisticRegression(max_iter=1000, random_state=42)
model_right.fit(X_train_scaled, y_train_r)
y_pred_right = model_right.predict(X_test_scaled)

acc_right = accuracy_score(y_test_r, y_pred_right)
f1_right = f1_score(y_test_r, y_pred_right)
print(f"RIGHT way (no leakage) - Accuracy: {acc_right:.4f}, F1: {f1_right:.4f}")

## 4. Comparison

In [ ]:
comparison = pd.DataFrame({
    'Method': ['Scale then Split (LEAKAGE)', 'Split then Scale (CORRECT)'],
    'Accuracy': [acc_wrong, acc_right],
    'F1 Score': [f1_wrong, f1_right]
}).round(4)

print(comparison.to_string(index=False))
print()
print("The difference may be small on this dataset, but on smaller datasets")
print("or with more preprocessing steps, leakage can inflate metrics significantly.")
print("The correct way ensures your reported metrics reflect real-world performance.")

## 5. Build an sklearn Pipeline

Pipelines guarantee that preprocessing steps are applied correctly -- fit on train, transform on test -- automatically.

We build a full pipeline that handles:
1. Imputation (numeric: median, categorical: most frequent)
2. Scaling (numeric)
3. Encoding (categorical: one-hot)
4. Model (logistic regression)

In [ ]:
# Define transformers for numeric and categorical columns
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

# Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, cat_cols)
    ]
)

# Full pipeline: preprocessor + model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

print("Pipeline created:")
print(pipeline)

In [ ]:
# Split and train -- the pipeline handles everything correctly
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline.fit(X_train, y_train)
y_pred_pipe = pipeline.predict(X_test)

acc_pipe = accuracy_score(y_test, y_pred_pipe)
f1_pipe = f1_score(y_test, y_pred_pipe)
print(f"Pipeline - Accuracy: {acc_pipe:.4f}, F1: {f1_pipe:.4f}")

## 6. TODO: Stratified Cross-Validation with the Pipeline

Cross-validation gives a more robust estimate of model performance. Your task:
1. Use `StratifiedKFold` with 5 folds
2. Use `cross_val_score` with the pipeline
3. Report mean and standard deviation of accuracy
4. Also compute cross-validated F1 scores

In [ ]:
# TODO: Implement stratified 5-fold cross-validation
#
# Hints:
# - cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# - accuracy_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='accuracy')
# - f1_scores = cross_val_score(pipeline, X, y, cv=cv, scoring='f1')
# - Print mean and std for both

# YOUR CODE HERE


---
## Summary

| Concept | Details |
|---------|---------|
| Data leakage | Information from test set leaks into training via preprocessing |
| Wrong way | Scale entire dataset, then split |
| Right way | Split first, fit scaler on train only |
| Pipeline | Automates correct preprocessing order |
| ColumnTransformer | Applies different transforms to numeric vs categorical columns |
| Cross-validation | More robust performance estimate than a single split |